# IMPORTAR PAQUETES

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from dotenv import load_dotenv
import janitor
import os
from pathlib import Path
import openpyxl
import sqlalchemy as sa

from sklearn.feature_selection import mutual_info_regression
from sklearn.feature_selection import RFE
from xgboost import XGBRegressor
from sklearn.inspection import permutation_importance
from sklearn.model_selection import TimeSeriesSplit
from sklearn.ensemble import HistGradientBoostingRegressor
from sklearn.pipeline import Pipeline
from sklearn.model_selection import RandomizedSearchCV
from sklearn.metrics import mean_absolute_error

# Desactivar notación científica
pd.set_option('display.float_format', lambda x: '%.3f' % x)
np.set_printoptions(suppress=True)

# Cargar variables de entorno
load_dotenv()

print("✅ Librerías importadas correctamente")

# EMPAQUETAR TODO EL PROCESO DE FUNCIONES

## Calidad de datos

In [ ]:
def calidad_datos(x, num=None):

    # Modificar los tipos
    temp = x.astype({'month': 'O', 'wday': 'O'})

    # Imputar los nulos
    temp.loc[x['event_name_1'].isna(), 'event_name_1'] = 'Sin_evento'

    def imputar_moda(registros):
        # Calcular la moda del precio en ese grupo semana-producto
        moda_serie = registros.sell_price.mode()
        if not moda_serie.empty:
            moda = moda_serie.iloc[0]
        else:
            # Si no hay moda en esa semana, calcular la moda a nivel de producto
            item_id = registros.item_id.iloc[0]
            if num is not None:
                moda_producto = num[num.item_id == item_id].sell_price.mode()
                if not moda_producto.empty:
                    moda = moda_producto.iloc[0]
                else:
                    moda = np.nan
            else:
                moda = np.nan

        registros.loc[registros.sell_price.isna(), 'sell_price'] = moda
        return registros

    temp = temp.groupby('item_id', group_keys=False).apply(imputar_moda)
    return temp

## Creación de variables

In [ ]:
def crear_variables(x):

    # DEMANDA INTERMITENTE

    def rotura_stock(ventas, n = 5):
        cero_ventas = pd.Series(np.where(ventas == 0, 1, 0))
        num_ceros = cero_ventas.rolling(n).sum()
        rotura_stock = np.where(num_ceros == n, 1, 0)
        return (rotura_stock)

    x = x.sort_values(by = ['store_id','item_id','date'])
    x['rotura_stock_3'] = x.groupby(['store_id', 'item_id']).ventas.transform(lambda x: rotura_stock(x, 3)).values
    x['rotura_stock_7'] = x.groupby(['store_id', 'item_id']).ventas.transform(lambda x: rotura_stock(x, 7)).values
    x['rotura_stock_15'] = x.groupby(['store_id', 'item_id']).ventas.transform(lambda x: rotura_stock(x, 15)).values

    # LAGS

    def crear_lags(df, variable, num_lags = 7):

        # Crear el objeto dataframe
        lags = pd.DataFrame()

        # Crea todos los lags 
        for cada in range(1, num_lags + 1):
            lags[variable + '_lag_' + str(cada)] = df[variable].shift(cada)

        # Devuelve el dataframe con los lags    
        return(lags)

    lags_sell_price_df = (x.groupby(['store_id', 'item_id'])
                        .apply(lambda x: crear_lags(x = x, variable = 'sell_price', num_lags= 7))
                        .reset_index()
                        .set_index('date'))

    lags_rotura_stock_3_df = (x.groupby(['store_id', 'item_id'])
                            .apply(lambda x: crear_lags(x = x, variable = 'rotura_stock_3', num_lags= 1))
                            .reset_index()
                            .set_index('date'))

    lags_rotura_stock_7_df = (x.groupby(['store_id', 'item_id'])
                                .apply(lambda x : crear_lags(x = x, variable= 'rotura_stock_7', num_lags= 1))
                                .reset_index()
                                .set_index('date'))

    lags_rotura_stock_15_df = (x.groupby(['store_id', 'item_id'])
                                .apply(lambda x : crear_lags(x = x, variable= 'rotura_stock_15', num_lags= 1))
                                .reset_index()
                                .set_index('date'))

    lags_ventas_df = (x.groupby(['store_id', 'item_id'])
                        .apply(lambda x : crear_lags(x = x, variable='ventas', num_lags= 15))
                        .reset_index()
                        .set_index('date'))


    # Ventanas MÓVILES

    def min_movil(df, variable, num_periodos= 7):

        minm = pd.DataFrame()

        for cada in range(2, num_periodos+1):
            minm[variable + '_minm_' + str(cada)] = df[variable].shift(1).rolling(cada).min()

        return(minm)


    def media_movil(df, variable, num_periodos= 7):

        mm = pd.DataFrame()

        for cada in range(2, num_periodos+1):
            mm[variable + '_mm_' + str(cada)] = df[variable].shift(1).rolling(cada).mean()

        return(mm)


    def max_movil(df, variable, num_periodos= 7):

        maxm = pd.DataFrame()

        for cada in range(2, num_periodos+1):
            maxm[variable + '_maxm_' + str(cada)] = df[variable].shift(1).rolling(cada).max()

        return(maxm)


    min_movil_df = (x.groupby(['store_id', 'item_id'])
                    .apply(lambda x : min_movil(df = x, variable= 'ventas', num_periodos= 15))
                    .reset_index()
                    .set_index('date'))

    media_movil_df = (x.groupby(['store_id', 'item_id'])
                    .apply(lambda x : media_movil(df = x, variable= 'ventas', num_periodos= 15))
                    .reset_index()
                    .set_index('date'))

    max_movil_df = (x.groupby(['store_id', 'item_id'])
                    .apply(lambda x : max_movil(df = x, variable= 'ventas', num_periodos= 15))
                    .reset_index()
                    .set_index('date'))

    # UNIR DATAFRAMES CREADOS

    x_unido = pd.concat([x,
             lags_sell_price_df,
             lags_rotura_stock_3_df,
             lags_rotura_stock_7_df,
             lags_rotura_stock_15_df,
             lags_ventas_df,
             min_movil_df,
             media_movil_df,
             max_movil_df], axis = 1 )

    # Eliminar columnas duplicadas
    x_unido = x_unido.loc[:,~x_unido.columns.duplicated()]

    x_unido.dropna(inplace=True)

    x_unido.drop(columns= ['sell_price', 'rotura_stock_3', 'rotura_stock_7', 'rotura_stock_15'], 
                 inplace=True)

    # Crear una sola variable para producto

    x_unido = x_unido.insert(loc=0, column='Producto', value=x_unido.store_id + ' ' + x_unido.item_id)

    x_unido = x_unido.drop(columns= ['store_id', 'item_id'])

    return (x_unido)

## Transformación de Variables